In [1]:
import numpy as np
import matplotlib.pyplot as plt

import tides

print("TIDES loaded from:")
print(tides.__file__)

TIDES loaded from:
E:\Code\Github_clone\TIDES-Inference\tides.py


In [2]:
# ------------------------------------------------------------
# N=8 benchmark truth
# ------------------------------------------------------------

N = 8
dt = 5e-4
T_total = 0.36

true_change_times = np.array([
    0.06,
    0.12,
    0.18,
    0.24,
    0.30,
])

true_transition_indices = np.rint(true_change_times / dt).astype(int)

print("True change times   :", true_change_times)
print("True transition idx :", true_transition_indices)

True change times   : [0.06 0.12 0.18 0.24 0.3 ]
True transition idx : [120 240 360 480 600]


In [3]:
# ============================================================
# Cell 3 — Pair-only canonical N=8 benchmark
# ============================================================

from itertools import combinations
import numpy as np

# ------------------------------------------------------------
# Basic system parameters
# ------------------------------------------------------------

SEED = 20260811

N = 8
Q_CONSERVED = 1
OBSERVABLE_CAPACITY = N - Q_CONSERVED

DT = 5.0e-4
STAGE_DURATION = 0.060
N_STAGES = 6

INTERVALS_PER_STAGE = int(round(STAGE_DURATION / DT))

assert INTERVALS_PER_STAGE == 120


# ------------------------------------------------------------
# Shared pairwise interaction law
#
# phi(x) = x + 0.5 x^2
#
# Edge interaction:
#
# phi(x_j) - phi(x_i)
# =
# (x_j - x_i)
# + 0.5 (x_j^2 - x_i^2)
#
# Therefore the true edge-space coefficient direction is
#
# theta = (1, 0.5)
# ------------------------------------------------------------

PAIR_LAW_TRUE = np.array(
    [1.0, 0.5],
    dtype=float
)


# ------------------------------------------------------------
# Candidate pairwise edge space
#
# Inference will consider ALL 28 undirected candidate edges.
# ------------------------------------------------------------

CANDIDATE_EDGES = tuple(
    combinations(range(1, N + 1), 2)
)

assert len(CANDIDATE_EDGES) == 28

EDGE_INDEX_1B = {
    edge: m
    for m, edge in enumerate(CANDIDATE_EDGES)
}


# ------------------------------------------------------------
# Fixed heterogeneous microscopic edge weights
#
# Each candidate edge has a deterministic weight.
# Only edges active in a given snapshot contribute.
# ------------------------------------------------------------

weight_rng = np.random.default_rng(SEED)

EDGE_WEIGHT = {
    edge: int(weight_rng.integers(800, 1201)) / 1000.0
    for edge in CANDIDATE_EDGES
}


# ------------------------------------------------------------
# Six connected pairwise snapshots
#
# Each stage:
#   - 10 active edges
#   - connected graph
#
# Each transition:
#   - exactly 4 changed supports
#   - 2 removals + 2 additions
#   - changed-edge set is a forest
# ------------------------------------------------------------

SNAPSHOTS = (
    (
        (1, 3), (2, 5), (2, 8), (3, 5), (3, 8),
        (4, 6), (4, 7), (5, 6), (6, 8), (7, 8),
    ),

    (
        (1, 3), (1, 5), (2, 3), (2, 8), (3, 5),
        (3, 8), (4, 6), (4, 7), (5, 6), (7, 8),
    ),

    (
        (1, 3), (1, 6), (2, 3), (2, 7), (2, 8),
        (3, 8), (4, 6), (4, 7), (5, 6), (7, 8),
    ),

    (
        (1, 3), (1, 4), (1, 6), (2, 7), (2, 8),
        (3, 8), (4, 5), (4, 6), (5, 6), (7, 8),
    ),

    (
        (1, 3), (1, 6), (2, 7), (2, 8), (3, 8),
        (4, 5), (5, 6), (5, 7), (6, 7), (7, 8),
    ),

    (
        (1, 3), (1, 8), (2, 8), (3, 8), (4, 5),
        (4, 8), (5, 6), (5, 7), (6, 7), (7, 8),
    ),
)


# ------------------------------------------------------------
# Initial condition
#
# Same representative state as the previous benchmark.
# Recentered so sum_i x_i = 0 exactly.
# ------------------------------------------------------------

X0 = np.array(
    [
        -0.319989,
        -0.301418,
         0.354093,
        -0.213312,
        -0.026615,
         0.067347,
         0.282008,
         0.157887,
    ],
    dtype=float,
)

X0 -= X0.mean()

assert abs(X0.sum()) < 1e-14


# ------------------------------------------------------------
# Pairwise vector field
# ------------------------------------------------------------

def phi(x):
    return x + 0.5 * x * x


def edge_field(x, edge):

    i, j = edge

    # convert 1-based node labels to Python indices
    i -= 1
    j -= 1

    out = np.zeros_like(x, dtype=float)

    flux = EDGE_WEIGHT[edge] * (
        phi(x[j]) - phi(x[i])
    )

    out[i] += flux
    out[j] -= flux

    return out


def stage_field(x, stage):

    out = np.zeros(N, dtype=float)

    for edge in SNAPSHOTS[stage]:
        out += edge_field(x, edge)

    return out


def rk4_step(x, dt, stage):

    k1 = stage_field(x, stage)

    k2 = stage_field(
        x + 0.5 * dt * k1,
        stage
    )

    k3 = stage_field(
        x + 0.5 * dt * k2,
        stage
    )

    k4 = stage_field(
        x + dt * k3,
        stage
    )

    return x + (
        dt / 6.0
    ) * (
        k1
        + 2.0 * k2
        + 2.0 * k3
        + k4
    )


# ------------------------------------------------------------
# Structural audits
# ------------------------------------------------------------

def is_connected(edge_set):

    adjacency = {
        i: set()
        for i in range(1, N + 1)
    }

    for i, j in edge_set:
        adjacency[i].add(j)
        adjacency[j].add(i)

    seen = {1}
    stack = [1]

    while stack:

        u = stack.pop()

        for v in adjacency[u]:

            if v not in seen:
                seen.add(v)
                stack.append(v)

    return len(seen) == N


def is_forest(edge_set):

    parent = {
        i: i
        for i in range(1, N + 1)
    }

    def find(a):

        while parent[a] != a:

            parent[a] = parent[parent[a]]
            a = parent[a]

        return a

    for a, b in edge_set:

        ra = find(a)
        rb = find(b)

        if ra == rb:
            return False

        parent[ra] = rb

    return True


transition_supports_true = []

for k in range(N_STAGES - 1):

    before = set(SNAPSHOTS[k])
    after = set(SNAPSHOTS[k + 1])

    changed = tuple(
        sorted(before ^ after)
    )

    transition_supports_true.append(changed)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert all(
    len(snapshot) == 10
    for snapshot in SNAPSHOTS
)

assert all(
    len(set(snapshot)) == len(snapshot)
    for snapshot in SNAPSHOTS
)

assert all(
    set(snapshot).issubset(CANDIDATE_EDGES)
    for snapshot in SNAPSHOTS
)

assert all(
    is_connected(snapshot)
    for snapshot in SNAPSHOTS
)

assert all(
    len(S) == 4
    for S in transition_supports_true
)

assert all(
    is_forest(S)
    for S in transition_supports_true
)

assert all(
    len(S) < OBSERVABLE_CAPACITY
    for S in transition_supports_true
)


# ------------------------------------------------------------
# Truth change times
# ------------------------------------------------------------

true_change_times = (
    np.arange(1, N_STAGES)
    * STAGE_DURATION
)

true_transition_indices = np.rint(
    true_change_times / DT
).astype(int)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

active_union = set().union(
    *map(set, SNAPSHOTS)
)

print("=" * 90)
print("TIDES N=8 pair-only canonical benchmark")
print("=" * 90)

print("N                       :", N)
print("candidate edges M       :", len(CANDIDATE_EDGES))
print("active edges / stage    :", [len(G) for G in SNAPSHOTS])
print("unique active edges     :", len(active_union))

print("conserved scalars       :", Q_CONSERVED)
print("observable capacity     :", OBSERVABLE_CAPACITY)

print("dt                      :", DT)
print("stage duration          :", STAGE_DURATION)
print("intervals / stage       :", INTERVALS_PER_STAGE)

print("pair law theta          :", PAIR_LAW_TRUE)

print("\nSnapshot connectivity:")

for r, G in enumerate(SNAPSHOTS, start=1):

    print(
        f"G{r}: "
        f"edges={len(G)}, "
        f"connected={is_connected(G)}"
    )

print("\nTransition structural load:")

for k, S in enumerate(
    transition_supports_true,
    start=1
):

    before = set(SNAPSHOTS[k - 1])
    after = set(SNAPSHOTS[k])

    removed = tuple(sorted(before - after))
    added = tuple(sorted(after - before))

    print(
        f"G{k}->G{k+1}: "
        f"s={len(S)}, "
        f"removed={len(removed)}, "
        f"added={len(added)}, "
        f"forest={is_forest(S)}, "
        f"changes={S}"
    )

print("\nTrue change times   :", true_change_times)
print("True transition idx :", true_transition_indices)

TIDES N=8 pair-only canonical benchmark
N                       : 8
candidate edges M       : 28
active edges / stage    : [10, 10, 10, 10, 10, 10]
unique active edges     : 20
conserved scalars       : 1
observable capacity     : 7
dt                      : 0.0005
stage duration          : 0.06
intervals / stage       : 120
pair law theta          : [1.  0.5]

Snapshot connectivity:
G1: edges=10, connected=True
G2: edges=10, connected=True
G3: edges=10, connected=True
G4: edges=10, connected=True
G5: edges=10, connected=True
G6: edges=10, connected=True

Transition structural load:
G1->G2: s=4, removed=2, added=2, forest=True, changes=((1, 5), (2, 3), (2, 5), (6, 8))
G2->G3: s=4, removed=2, added=2, forest=True, changes=((1, 5), (1, 6), (2, 7), (3, 5))
G3->G4: s=4, removed=2, added=2, forest=True, changes=((1, 4), (2, 3), (4, 5), (4, 7))
G4->G5: s=4, removed=2, added=2, forest=True, changes=((1, 4), (4, 6), (5, 7), (6, 7))
G5->G6: s=4, removed=2, added=2, forest=True, changes=((1, 6),

In [4]:
# ============================================================
# Cell 4 — Generate pair-only benchmark trajectory
# ============================================================

n_total_intervals = (
    N_STAGES
    * INTERVALS_PER_STAGE
)

T = (
    np.arange(
        n_total_intervals + 1,
        dtype=float
    )
    * DT
)

X = np.empty(
    (n_total_intervals + 1, N),
    dtype=float
)

X[0] = X0


# ------------------------------------------------------------
# True stage label for each integration interval
# ------------------------------------------------------------

true_stage_of_interval = np.repeat(
    np.arange(
        N_STAGES,
        dtype=int
    ),
    INTERVALS_PER_STAGE,
)

assert len(true_stage_of_interval) == n_total_intervals


# ------------------------------------------------------------
# RK4 integration
# ------------------------------------------------------------

for n, stage in enumerate(
    true_stage_of_interval
):

    X[n + 1] = rk4_step(
        X[n],
        DT,
        int(stage)
    )


t = T.copy()


# ------------------------------------------------------------
# Basic trajectory diagnostics
# ------------------------------------------------------------

conservation_drift = np.max(
    np.abs(
        X.sum(axis=1)
        - X[0].sum()
    )
)

state_displacement = np.linalg.norm(
    X[-1] - X[0]
)

max_abs_state = np.max(
    np.abs(X)
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("=" * 90)
print("Cell 4 — pair-only trajectory generated")
print("=" * 90)

print(
    "state samples          :",
    len(X)
)

print(
    "trajectory intervals   :",
    len(X) - 1
)

print(
    "total duration         :",
    f"{t[-1]:.6f}"
)

print(
    "X shape                :",
    X.shape
)

print(
    "||X(T)-X(0)||          :",
    f"{state_displacement:.6e}"
)

print(
    "max |x_i(t)|           :",
    f"{max_abs_state:.6e}"
)

print(
    "max conservation drift :",
    f"{conservation_drift:.6e}"
)

print(
    "all finite             :",
    np.isfinite(X).all()
)


# ------------------------------------------------------------
# Regression assertions
# ------------------------------------------------------------

assert X.shape == (721, 8)
assert t.shape == (721,)

assert np.allclose(
    np.diff(t),
    DT
)

assert np.isfinite(X).all()

assert conservation_drift < 1e-12

assert np.array_equal(
    true_transition_indices,
    np.array(
        [120, 240, 360, 480, 600]
    )
)

Cell 4 — pair-only trajectory generated
state samples          : 721
trajectory intervals   : 720
total duration         : 0.360000
X shape                : (721, 8)
||X(T)-X(0)||          : 4.567773e-01
max |x_i(t)|           : 3.540929e-01
max conservation drift : 1.276756e-15
all finite             : True


In [5]:
# ============================================================
# Cell 5 — Formal TIDES Step 1: change-point detection
# ============================================================

# ------------------------------------------------------------
# A. Controlled-count regression
#
# This checks whether the Step-1 implementation places the
# five strongest change points at the correct samples.
# ------------------------------------------------------------

s1_controlled = tides.detect_changes(
    X,
    t,
    method="secant",
    n_changes=5,
    min_separation=10,
)

print("=" * 90)
print("TIDES Step 1 — controlled-count regression")
print("=" * 90)

print(
    "true indices       :",
    true_transition_indices
)

print(
    "detected indices   :",
    s1_controlled.transition_indices
)

print(
    "index errors       :",
    s1_controlled.transition_indices
    - true_transition_indices
)

print(
    "true times         :",
    true_change_times
)

print(
    "detected times     :",
    s1_controlled.transition_times
)


# ------------------------------------------------------------
# B. Fully blind threshold
#
# Same robust rule as the previous benchmark:
#
# threshold = median(chi) + 20 MAD(chi)
#
# We calculate it explicitly and pass it to the TIDES module.
# ------------------------------------------------------------

V_SECANT = np.diff(
    X,
    axis=0
) / DT

jump = np.linalg.norm(
    V_SECANT[1:]
    - V_SECANT[:-1],
    axis=1
)

jump_median = np.median(jump)

jump_mad = np.median(
    np.abs(
        jump - jump_median
    )
)

blind_threshold = (
    jump_median
    + 20.0 * jump_mad
)


s1_blind = tides.detect_changes(
    X,
    t,
    method="secant",
    threshold=blind_threshold,
    min_separation=1,
)


print("\n" + "=" * 90)
print("TIDES Step 1 — fully blind detection")
print("=" * 90)

print(
    "jump median        :",
    f"{jump_median:.6e}"
)

print(
    "jump MAD           :",
    f"{jump_mad:.6e}"
)

print(
    "blind threshold    :",
    f"{blind_threshold:.6e}"
)

print(
    "detected count     :",
    s1_blind.n_transitions
)

print(
    "detected indices   :",
    s1_blind.transition_indices
)

print(
    "detected times     :",
    s1_blind.transition_times
)


# ------------------------------------------------------------
# Regression verdict
# ------------------------------------------------------------

controlled_exact = np.array_equal(
    s1_controlled.transition_indices,
    true_transition_indices
)

blind_exact = np.array_equal(
    s1_blind.transition_indices,
    true_transition_indices
)

print("\n" + "=" * 90)

print(
    "controlled exact   :",
    controlled_exact
)

print(
    "blind exact        :",
    blind_exact
)

if controlled_exact and blind_exact:
    print("TIDES STEP 1: PASS")
else:
    print("TIDES STEP 1: NEEDS INSPECTION")

print("=" * 90)

TIDES Step 1 — controlled-count regression
true indices       : [120 240 360 480 600]
detected indices   : [120 240 360 480 600]
index errors       : [0 0 0 0 0]
true times         : [0.06 0.12 0.18 0.24 0.3 ]
detected times     : [0.06 0.12 0.18 0.24 0.3 ]

TIDES Step 1 — fully blind detection
jump median        : 2.568056e-03
jump MAD           : 8.507082e-04
blind threshold    : 1.958222e-02
detected count     : 5
detected indices   : [120 240 360 480 600]
detected times     : [0.06 0.12 0.18 0.24 0.3 ]

controlled exact   : True
blind exact        : True
TIDES STEP 1: PASS


In [6]:
# ============================================================
# Cell 6 — PREPROCESSING
#          Segment trajectory and reconstruct midpoint states
#          and velocities from Step-1 detected change points
# ============================================================

# ------------------------------------------------------------
# IMPORTANT:
#
# From this point onward, segmentation is determined only by
# the BLIND Step-1 output.
#
# No true transition times / indices are used here.
# ------------------------------------------------------------

detected_transition_indices = (
    s1_blind.transition_indices.copy()
)

detected_transition_times = (
    s1_blind.transition_times.copy()
)


# ------------------------------------------------------------
# 1. Interval secant velocities
#
# V_SECANT[n] corresponds to interval [t_n, t_{n+1}]
# ------------------------------------------------------------

V_SECANT = np.diff(
    X,
    axis=0
) / DT

n_intervals = len(V_SECANT)


# ------------------------------------------------------------
# 2. Build data-derived temporal segments
#
# For five detected transitions:
#
# [0,120), [120,240), ..., [600,720)
# ------------------------------------------------------------

segment_bounds = np.concatenate(
    (
        [0],
        detected_transition_indices,
        [n_intervals],
    )
)

segment_slices = tuple(
    slice(
        int(segment_bounds[k]),
        int(segment_bounds[k + 1]),
    )
    for k in range(
        len(segment_bounds) - 1
    )
)

segment_interval_counts = np.array(
    [
        sl.stop - sl.start
        for sl in segment_slices
    ],
    dtype=int,
)


# ------------------------------------------------------------
# 3. Stage label for every trajectory interval
#
# This is inferred entirely from Step 1.
# ------------------------------------------------------------

stage_of_interval = np.empty(
    n_intervals,
    dtype=int,
)

for stage, sl in enumerate(segment_slices):
    stage_of_interval[sl] = stage


# ------------------------------------------------------------
# 4. High-order midpoint reconstruction
#
# For interval n, midpoint t_{n+1/2}:
#
# X_mid =
# (-X[n-1] + 9 X[n] + 9 X[n+1] - X[n+2]) / 16
#
# V_mid =
# (X[n-1] - 27 X[n] + 27 X[n+1] - X[n+2])
# / (24 dt)
#
# We only retain samples whose full stencil lies inside the
# same detected stage.
# ------------------------------------------------------------

obs_interval_indices = []

X_MID = []
V_MID = []
OBS_STAGE = []
T_MID = []


for n in range(
    1,
    n_intervals - 1
):

    same_stage = (
        stage_of_interval[n - 1]
        == stage_of_interval[n]
        == stage_of_interval[n + 1]
    )

    if not same_stage:
        continue

    x_mid = (
        -X[n - 1]
        + 9.0 * X[n]
        + 9.0 * X[n + 1]
        - X[n + 2]
    ) / 16.0

    v_mid = (
        X[n - 1]
        - 27.0 * X[n]
        + 27.0 * X[n + 1]
        - X[n + 2]
    ) / (
        24.0 * DT
    )

    t_mid = (
        t[n] + t[n + 1]
    ) / 2.0

    obs_interval_indices.append(n)
    X_MID.append(x_mid)
    V_MID.append(v_mid)
    T_MID.append(t_mid)

    OBS_STAGE.append(
        stage_of_interval[n]
    )


# ------------------------------------------------------------
# 5. Convert to arrays
# ------------------------------------------------------------

obs_interval_indices = np.asarray(
    obs_interval_indices,
    dtype=int,
)

X_MID = np.asarray(
    X_MID,
    dtype=float,
)

V_MID = np.asarray(
    V_MID,
    dtype=float,
)

T_MID = np.asarray(
    T_MID,
    dtype=float,
)

OBS_STAGE = np.asarray(
    OBS_STAGE,
    dtype=int,
)


# ------------------------------------------------------------
# 6. Store preprocessing output
#
# This is what later Step 2 / Step 3 will consume.
# ------------------------------------------------------------

TIDES_OBS = {
    "t_state": t.copy(),
    "x_state": X.copy(),

    "detected_transition_indices":
        detected_transition_indices.copy(),

    "detected_transition_times":
        detected_transition_times.copy(),

    "segment_bounds":
        segment_bounds.copy(),

    "segment_interval_counts":
        segment_interval_counts.copy(),

    "obs_interval_indices":
        obs_interval_indices.copy(),

    "t_mid":
        T_MID.copy(),

    "x_mid":
        X_MID.copy(),

    "velocity_mid":
        V_MID.copy(),

    "stage_of_observation":
        OBS_STAGE.copy(),
}


# ------------------------------------------------------------
# 7. Diagnostics
# ------------------------------------------------------------

samples_per_stage = np.bincount(
    OBS_STAGE,
    minlength=N_STAGES
)

print("=" * 90)
print("Cell 6 — PREPROCESSING")
print("=" * 90)

print(
    "detected transitions   :",
    detected_transition_indices
)

print(
    "segment lengths        :",
    segment_interval_counts.tolist()
)

print(
    "inference observations :",
    len(X_MID)
)

print(
    "samples / stage        :",
    samples_per_stage.tolist()
)

print(
    "X_MID shape            :",
    X_MID.shape
)

print(
    "V_MID shape            :",
    V_MID.shape
)

print(
    "T_MID shape            :",
    T_MID.shape
)

print(
    "all midpoint states finite :",
    np.isfinite(X_MID).all()
)

print(
    "all midpoint velocities finite :",
    np.isfinite(V_MID).all()
)


# ------------------------------------------------------------
# 8. Regression assertions
# ------------------------------------------------------------

assert len(segment_slices) == N_STAGES

assert np.array_equal(
    segment_interval_counts,
    np.full(
        N_STAGES,
        120,
        dtype=int,
    )
)

assert X_MID.shape == (708, N)
assert V_MID.shape == (708, N)
assert T_MID.shape == (708,)

assert np.array_equal(
    samples_per_stage,
    np.full(
        N_STAGES,
        118,
        dtype=int,
    )
)

assert np.isfinite(X_MID).all()
assert np.isfinite(V_MID).all()

print("\nPREPROCESSING: PASS")

Cell 6 — PREPROCESSING
detected transitions   : [120 240 360 480 600]
segment lengths        : [120, 120, 120, 120, 120, 120]
inference observations : 708
samples / stage        : [118, 118, 118, 118, 118, 118]
X_MID shape            : (708, 8)
V_MID shape            : (708, 8)
T_MID shape            : (708,)
all midpoint states finite : True
all midpoint velocities finite : True

PREPROCESSING: PASS


In [7]:
# ============================================================
# Cell 7 — PREPROCESSING DIAGNOSTIC
#          Resolution audit: 4-point vs 6-point midpoint
#          reconstruction
# ============================================================

import math


# ------------------------------------------------------------
# 1. Generic finite-difference / interpolation weights
#
# We compare:
#
#   4-point stencil:
#       offsets = [-3/2, -1/2, +1/2, +3/2]
#
#   6-point stencil:
#       offsets = [-5/2, -3/2, -1/2,
#                  +1/2, +3/2, +5/2]
#
# Both reconstruct the state and velocity at the midpoint.
# ------------------------------------------------------------

def finite_difference_weights(
    nodes,
    derivative_order,
):

    nodes = np.asarray(
        nodes,
        dtype=float,
    )

    n = len(nodes)

    A = np.vstack([
        nodes**k
        for k in range(n)
    ])

    b = np.zeros(
        n,
        dtype=float,
    )

    b[derivative_order] = math.factorial(
        derivative_order
    )

    return np.linalg.solve(A, b)


z4 = np.array(
    [-1.5, -0.5, 0.5, 1.5],
    dtype=float,
)

z6 = np.array(
    [-2.5, -1.5, -0.5, 0.5, 1.5, 2.5],
    dtype=float,
)

w4_x = finite_difference_weights(
    z4,
    derivative_order=0,
)

w4_d = finite_difference_weights(
    z4,
    derivative_order=1,
)

w6_x = finite_difference_weights(
    z6,
    derivative_order=0,
)

w6_d = finite_difference_weights(
    z6,
    derivative_order=1,
)


# ------------------------------------------------------------
# 2. Compare 4-point and 6-point reconstructions
#
# Only use midpoint locations whose entire 6-point stencil
# remains within one Step-1-derived temporal stage.
#
# No oracle transition information is used.
# ------------------------------------------------------------

rel_state_46 = []
rel_velocity_46 = []

abs_state_46 = []
abs_velocity_46 = []


for n in range(
    2,
    n_intervals - 2,
):

    # Six-point stencil around midpoint n+1/2 touches
    # intervals n-2, ..., n+2.
    stencil_stages = {
        stage_of_interval[n + j]
        for j in (-2, -1, 0, 1, 2)
    }

    if len(stencil_stages) != 1:
        continue

    p4 = X[n - 1 : n + 3]
    p6 = X[n - 2 : n + 4]

    x4 = (
        w4_x[:, None]
        * p4
    ).sum(axis=0)

    x6 = (
        w6_x[:, None]
        * p6
    ).sum(axis=0)

    v4 = (
        w4_d[:, None]
        * p4
    ).sum(axis=0) / DT

    v6 = (
        w6_d[:, None]
        * p6
    ).sum(axis=0) / DT

    dx = np.linalg.norm(
        x6 - x4
    )

    dv = np.linalg.norm(
        v6 - v4
    )

    rel_state_46.append(
        dx
        / max(
            np.linalg.norm(x6),
            1e-15,
        )
    )

    rel_velocity_46.append(
        dv
        / max(
            np.linalg.norm(v6),
            1e-15,
        )
    )

    abs_state_46.append(dx)
    abs_velocity_46.append(dv)


rel_state_46 = np.asarray(
    rel_state_46
)

rel_velocity_46 = np.asarray(
    rel_velocity_46
)

abs_state_46 = np.asarray(
    abs_state_46
)

abs_velocity_46 = np.asarray(
    abs_velocity_46
)


# ------------------------------------------------------------
# 3. Data-derived numerical resolution floor
#
# Keep the same conservative convention used previously:
#
# PROFILE_FLOOR =
#     20 × max relative 4-vs-6 discrepancy
#
# This is NOT a statistical noise estimate.
# It is a numerical-resolution diagnostic.
# ------------------------------------------------------------

resolution_rel_max = max(
    float(rel_state_46.max()),
    float(rel_velocity_46.max()),
    1e-14,
)

PROFILE_FLOOR = (
    20.0
    * resolution_rel_max
)


# ------------------------------------------------------------
# 4. Store diagnostic
# ------------------------------------------------------------

TIDES_OBS.update({
    "rel_state_4v6":
        rel_state_46.copy(),

    "rel_velocity_4v6":
        rel_velocity_46.copy(),

    "resolution_rel_max":
        resolution_rel_max,

    "profile_floor":
        PROFILE_FLOOR,
})


# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

print("=" * 90)
print("Cell 7 — PREPROCESSING DIAGNOSTIC")
print("=" * 90)

print(
    "resolution samples       :",
    len(rel_state_46)
)

print(
    "max relative state 4-vs-6:",
    f"{rel_state_46.max():.6e}"
)

print(
    "max relative vel. 4-vs-6 :",
    f"{rel_velocity_46.max():.6e}"
)

print(
    "median relative velocity :",
    f"{np.median(rel_velocity_46):.6e}"
)

print(
    "resolution relative max  :",
    f"{resolution_rel_max:.6e}"
)

print(
    "data-derived profile floor:",
    f"{PROFILE_FLOOR:.6e}"
)


# ------------------------------------------------------------
# 6. Sanity checks
# ------------------------------------------------------------

assert len(rel_state_46) > 0
assert np.isfinite(rel_state_46).all()
assert np.isfinite(rel_velocity_46).all()

assert PROFILE_FLOOR > 0.0

print("\nPREPROCESSING DIAGNOSTIC: PASS")

Cell 7 — PREPROCESSING DIAGNOSTIC
resolution samples       : 696
max relative state 4-vs-6: 1.205661e-12
max relative vel. 4-vs-6 : 8.764886e-13
median relative velocity : 4.979685e-13
resolution relative max  : 1.205661e-12
data-derived profile floor: 2.411322e-11

PREPROCESSING DIAGNOSTIC: PASS


In [8]:
# ============================================================
# Cell 8 — TIDES STEP 2 (BLIND)
#          Build the edge-space temporal change dictionary
#          and eliminate the stationary anchor nuisance
# ============================================================

# ------------------------------------------------------------
# IMPORTANT:
#
# Inference-side information used from this point:
#
#   X_MID, V_MID, OBS_STAGE
#       <- obtained from observable trajectory + blind Step 1
#
#   candidate edge set
#       <- model specification
#
#   pair-function library
#       psi_1(x_i,x_j) = x_j - x_i
#       psi_2(x_i,x_j) = x_j^2 - x_i^2
#
# NO true snapshots, true changed supports, true edge weights,
# or true theta are used below.
# ------------------------------------------------------------


# ------------------------------------------------------------
# 1. Candidate edge-space geometry
# ------------------------------------------------------------

M = len(CANDIDATE_EDGES)
L = 2
K = len(detected_transition_indices)

assert M == 28
assert L == 2
assert K == 5


# D[:,m] maps the scalar response on candidate edge m
# back to node space.
#
# Convention matches the forward benchmark:
# edge (i,j) -> + at i, - at j.
D = np.zeros(
    (N, M),
    dtype=float,
)

for m, (i, j) in enumerate(CANDIDATE_EDGES):

    D[i - 1, m] = +1.0
    D[j - 1, m] = -1.0


print(
    "candidate edges M      :",
    M
)

print(
    "incidence rank         :",
    np.linalg.matrix_rank(D)
)


# ------------------------------------------------------------
# 2. Edge-local function library
#
# EDGE_PSI[t,m,l]
#
# l = 0:
#     psi_1 = x_j - x_i
#
# l = 1:
#     psi_2 = x_j^2 - x_i^2
# ------------------------------------------------------------

n_obs = len(X_MID)

EDGE_PSI = np.zeros(
    (n_obs, M, L),
    dtype=float,
)

for m, (i, j) in enumerate(CANDIDATE_EDGES):

    xi = X_MID[:, i - 1]
    xj = X_MID[:, j - 1]

    EDGE_PSI[:, m, 0] = (
        xj - xi
    )

    EDGE_PSI[:, m, 1] = (
        xj**2 - xi**2
    )


print(
    "EDGE_PSI shape         :",
    EDGE_PSI.shape
)


# ------------------------------------------------------------
# 3. Node-space pair basis
#
# PAIR_BASIS[t,n,m,l]
#
# = D[n,m] * psi_l(x_i,x_j)
#
# This is the linear response associated with one coefficient
# B_{m,l}.
# ------------------------------------------------------------

PAIR_BASIS = (
    D[None, :, :, None]
    * EDGE_PSI[:, None, :, :]
)

print(
    "PAIR_BASIS shape       :",
    PAIR_BASIS.shape
)


# ------------------------------------------------------------
# 4. Stationary anchor design
#
# B^(1) is unknown and unpenalized.
#
# There are:
#
#     M * L = 28 * 2 = 56
#
# stationary anchor coefficients.
# ------------------------------------------------------------

X_BASE = PAIR_BASIS.reshape(
    n_obs * N,
    M * L,
)

Y_STEP2 = V_MID.reshape(-1)


print(
    "stationary columns     :",
    X_BASE.shape[1]
)

print(
    "X_BASE shape           :",
    X_BASE.shape
)


# ------------------------------------------------------------
# 5. Temporal change dictionary
#
# For transition k and candidate edge m:
#
#     Delta B^(k)_{m,:}
#
# contains L=2 coefficients.
#
# Its column is active only for observations AFTER transition k.
#
# Therefore:
#
#     5 transitions
#   x 28 candidate edges
#   = 140 candidate change groups
#
# each group has 2 coefficients.
# ------------------------------------------------------------

change_columns = []
CHANGE_GROUP_LABELS = []

for k in range(K):

    # Delta B^(k) contributes to stages k+1, k+2, ...
    active = (
        OBS_STAGE >= (k + 1)
    )

    for m in range(M):

        group_columns = []

        for l in range(L):

            col = np.zeros(
                (n_obs, N),
                dtype=float,
            )

            col[active] = (
                PAIR_BASIS[
                    active,
                    :,
                    m,
                    l,
                ]
            )

            group_columns.append(
                col.reshape(-1)
            )

        change_columns.extend(
            group_columns
        )

        CHANGE_GROUP_LABELS.append(
            (k, m)
        )


X_CHANGE = np.column_stack(
    change_columns
)


N_CHANGE_GROUPS = len(
    CHANGE_GROUP_LABELS
)


print(
    "candidate change groups:",
    N_CHANGE_GROUPS
)

print(
    "coefficients / group   :",
    L
)

print(
    "temporal coefficients  :",
    X_CHANGE.shape[1]
)

print(
    "X_CHANGE shape         :",
    X_CHANGE.shape
)


# ------------------------------------------------------------
# 6. Eliminate the stationary anchor nuisance
#
# Step 2 only wants the change structure.
#
# We therefore partial out the unpenalized stationary anchor:
#
#     Y_perp =
#         (I - P_base) Y
#
#     X_change_perp =
#         (I - P_base) X_change
#
# using a numerically stable QR projection.
#
# This is exact nuisance elimination; no truth is used.
# ------------------------------------------------------------

Q_BASE, R_BASE = np.linalg.qr(
    X_BASE,
    mode="reduced",
)

base_rank = np.linalg.matrix_rank(
    R_BASE
)

Y_PERP = (
    Y_STEP2
    - Q_BASE @ (
        Q_BASE.T @ Y_STEP2
    )
)

X_CHANGE_PERP = (
    X_CHANGE
    - Q_BASE @ (
        Q_BASE.T @ X_CHANGE
    )
)


print(
    "stationary design rank :",
    base_rank,
    "/",
    X_BASE.shape[1]
)

print(
    "||Y||                 :",
    f"{np.linalg.norm(Y_STEP2):.6e}"
)

print(
    "||Y_perp||            :",
    f"{np.linalg.norm(Y_PERP):.6e}"
)


# ------------------------------------------------------------
# 7. Group indexing
#
# Group g contains the two consecutive coefficients:
#
#     [2g, 2g+1]
#
# corresponding to one (transition, edge) row of Delta B.
# ------------------------------------------------------------

CHANGE_GROUP_COLUMNS = tuple(
    np.array(
        [
            L * g + l
            for l in range(L)
        ],
        dtype=int,
    )
    for g in range(N_CHANGE_GROUPS)
)


# ------------------------------------------------------------
# 8. Store formal Step-2 design
# ------------------------------------------------------------

TIDES_STEP2_DESIGN = {
    "D":
        D.copy(),

    "edge_psi":
        EDGE_PSI.copy(),

    "pair_basis":
        PAIR_BASIS.copy(),

    "X_base":
        X_BASE.copy(),

    "X_change":
        X_CHANGE.copy(),

    "y":
        Y_STEP2.copy(),

    "Q_base":
        Q_BASE.copy(),

    "y_perp":
        Y_PERP.copy(),

    "X_change_perp":
        X_CHANGE_PERP.copy(),

    "group_labels":
        tuple(CHANGE_GROUP_LABELS),

    "group_columns":
        CHANGE_GROUP_COLUMNS,

    "profile_floor":
        PROFILE_FLOOR,
}


# ------------------------------------------------------------
# 9. Structural assertions
# ------------------------------------------------------------

assert D.shape == (8, 28)

assert EDGE_PSI.shape == (
    708,
    28,
    2,
)

assert PAIR_BASIS.shape == (
    708,
    8,
    28,
    2,
)

assert X_BASE.shape == (
    5664,
    56,
)

assert X_CHANGE.shape == (
    5664,
    280,
)

assert X_CHANGE_PERP.shape == (
    5664,
    280,
)

assert N_CHANGE_GROUPS == 140

assert all(
    len(cols) == 2
    for cols in CHANGE_GROUP_COLUMNS
)

assert base_rank == 56


print("\n" + "=" * 90)
print("Cell 8 — TIDES STEP 2 (BLIND)")
print("=" * 90)

print(
    "candidate edges        :",
    M
)

print(
    "candidate transitions  :",
    K
)

print(
    "candidate groups       :",
    N_CHANGE_GROUPS
)

print(
    "group dimension        :",
    L
)

print(
    "unpenalized anchor dim :",
    X_BASE.shape[1]
)

print(
    "penalized change dim   :",
    X_CHANGE.shape[1]
)

print(
    "profile floor          :",
    f"{PROFILE_FLOOR:.6e}"
)

print(
    "oracle information used:",
    "NO"
)

print("\nSTEP 2 DESIGN CONSTRUCTION: PASS")

candidate edges M      : 28
incidence rank         : 7
EDGE_PSI shape         : (708, 28, 2)
PAIR_BASIS shape       : (708, 8, 28, 2)
stationary columns     : 56
X_BASE shape           : (5664, 56)
candidate change groups: 140
coefficients / group   : 2
temporal coefficients  : 280
X_CHANGE shape         : (5664, 280)
stationary design rank : 56 / 56
||Y||                 : 3.774987e+01
||Y_perp||            : 5.506967e+00

Cell 8 — TIDES STEP 2 (BLIND)
candidate edges        : 28
candidate transitions  : 5
candidate groups       : 140
group dimension        : 2
unpenalized anchor dim : 56
penalized change dim   : 280
profile floor          : 2.411322e-11
oracle information used: NO

STEP 2 DESIGN CONSTRUCTION: PASS


In [10]:
# ============================================================
# Cell 9 — TIDES STEP 2:
# DENSE vs SCALABLE BACKEND REGRESSION
#
# Both backends solve the same Step-2 inference task.
#
# dense:
#     explicit cumulative change design
#     dense baseline projection
#     full grouped BPDN
#
# scalable:
#     sparse stationary anchor
#     lazy projected group blocks
#     conditional screening
#     restricted grouped BPDN
#
# Both use the SAME:
#
#     uncertainty floor
#     convex group ranking principle
#     ranked-prefix floor certification
#
# No oracle support / topology / edge weights / theta are used.
# ============================================================

import importlib
import step2_change_structure

importlib.reload(step2_change_structure)

from step2_change_structure import (
    infer_change_structure_from_observations,
)


# ------------------------------------------------------------
# 1. Transition metadata from blind Step 1
# ------------------------------------------------------------

DETECTED_TRANSITION_INDICES = np.asarray(
    s1_blind.transition_indices,
    dtype=int,
)

DETECTED_TRANSITION_TIMES = t[
    DETECTED_TRANSITION_INDICES
]


# ------------------------------------------------------------
# 2. Dense correctness/reference backend
# ------------------------------------------------------------

print("=" * 90)
print("STEP 2 — DENSE REFERENCE")
print("=" * 90)

STEP2_DENSE = infer_change_structure_from_observations(
    Y=V_MID,
    D=D,
    edge_features=EDGE_PSI,
    stage_of_sample=OBS_STAGE,

    hypothesis="varying_structure",

    transition_indices=DETECTED_TRANSITION_INDICES,
    transition_times=DETECTED_TRANSITION_TIMES,

    edge_labels=CANDIDATE_EDGES,

    uncertainty_floor=PROFILE_FLOOR,

    solver_method="group_bpdn_prefix",
    backend="dense",

    solver_kwargs={
        "max_iter": 10000,
        "tol": 1e-8,
        "check_every": 50,
        "verbose": True,
    },

    refit_method="dense_lstsq",

    require_solver_convergence=True,
    require_floor_reached=True,

    return_design=False,
)


# ------------------------------------------------------------
# 3. Scalable backend
# ------------------------------------------------------------

print()
print("=" * 90)
print("STEP 2 — SCALABLE BACKEND")
print("=" * 90)

STEP2_SCALABLE = infer_change_structure_from_observations(
    Y=V_MID,
    D=D,
    edge_features=EDGE_PSI,
    stage_of_sample=OBS_STAGE,

    hypothesis="varying_structure",

    transition_indices=DETECTED_TRANSITION_INDICES,
    transition_times=DETECTED_TRANSITION_TIMES,

    edge_labels=CANDIDATE_EDGES,

    uncertainty_floor=PROFILE_FLOOR,

    solver_method="group_bpdn_prefix",
    backend="scalable",

    solver_kwargs={
        "verbose": True,

        # Numerical tolerance for the restricted dense BPDN solve.
        # This is intentionally backend-specific.
        "restricted_solver_kwargs": {
            "max_iter": 10000,
            "tol": 1e-7,
            "check_every": 50,
        },
    },

    refit_method="dense_lstsq",

    require_solver_convergence=True,
    require_floor_reached=True,

    return_design=False,
)


# ------------------------------------------------------------
# 4. Backend comparison
# ------------------------------------------------------------

dense_groups = tuple(
    STEP2_DENSE.selected_groups
)

scalable_groups = tuple(
    STEP2_SCALABLE.selected_groups
)

dense_set = set(dense_groups)
scalable_set = set(scalable_groups)

same_support = (
    dense_set == scalable_set
)

dense_prefix = (
    STEP2_DENSE.selection_result.selected_prefix_size
)

scalable_prefix = (
    STEP2_SCALABLE.selection_result.selected_prefix_size
)


print()
print("=" * 90)
print("Cell 9 — DENSE / SCALABLE REGRESSION")
print("=" * 90)

print(
    "dense backend used          :",
    STEP2_DENSE.metadata[
        "computational_backend_used"
    ],
)

print(
    "scalable backend used       :",
    STEP2_SCALABLE.metadata[
        "computational_backend_used"
    ],
)

print(
    "dense selected groups       :",
    len(dense_groups),
)

print(
    "scalable selected groups    :",
    len(scalable_groups),
)

print(
    "dense selected prefix       :",
    dense_prefix,
)

print(
    "scalable selected prefix    :",
    scalable_prefix,
)

print(
    "dense final residual        :",
    f"{STEP2_DENSE.relative_residual:.6e}",
)

print(
    "scalable final residual     :",
    f"{STEP2_SCALABLE.relative_residual:.6e}",
)

print(
    "uncertainty floor           :",
    f"{PROFILE_FLOOR:.6e}",
)

print(
    "same final support          :",
    same_support,
)

print(
    "dense full design built     :",
    STEP2_DENSE.metadata[
        "global_change_design_materialized"
    ],
)

print(
    "scalable full design built  :",
    STEP2_SCALABLE.metadata[
        "global_change_design_materialized"
    ],
)

print(
    "estimated dense bytes       :",
    STEP2_SCALABLE.metadata[
        "estimated_explicit_dense_bytes"
    ],
)

if "screening_group_count" in STEP2_SCALABLE.metadata:

    print(
        "scalable screening groups  :",
        STEP2_SCALABLE.metadata[
            "screening_group_count"
        ],
    )

    print(
        "scalable screening residual:",
        f"{STEP2_SCALABLE.metadata['screening_relative_residual']:.6e}",
    )


# ------------------------------------------------------------
# 5. Transition-wise support comparison
# ------------------------------------------------------------

print()
print("-" * 90)
print("TRANSITION-WISE SUPPORTS")
print("-" * 90)

for dense_c, scalable_c in zip(
    STEP2_DENSE.constraints,
    STEP2_SCALABLE.constraints,
):

    print(
        f"transition {dense_c.transition_ordinal + 1} "
        f"@ t={dense_c.transition_time:.6f}"
    )

    print(
        "  dense     :",
        dense_c.support_labels,
    )

    print(
        "  scalable  :",
        scalable_c.support_labels,
    )


# ------------------------------------------------------------
# 6. Regression assertions
#
# These compare two inference backends only.
# No benchmark truth is used.
# ------------------------------------------------------------

assert (
    STEP2_DENSE.metadata[
        "computational_backend_used"
    ]
    == "dense"
)

assert (
    STEP2_SCALABLE.metadata[
        "computational_backend_used"
    ]
    == "scalable"
)

assert STEP2_DENSE.metadata[
    "global_change_design_materialized"
]

assert not STEP2_SCALABLE.metadata[
    "global_change_design_materialized"
]

assert STEP2_DENSE.relative_residual <= PROFILE_FLOOR
assert STEP2_SCALABLE.relative_residual <= PROFILE_FLOOR

assert same_support

assert dense_prefix == scalable_prefix


# ------------------------------------------------------------
# 7. Production result for downstream Steps 3–4
#
# From this point onward use the scalable result.
# ------------------------------------------------------------

STEP2_RESULT = STEP2_SCALABLE


print()
print(
    "oracle information used      : NO"
)

print()
print(
    "TIDES STEP 2 DENSE/SCALABLE REGRESSION: PASS"
)

STEP 2 — DENSE REFERENCE
Starting grouped basis-pursuit denoising...
  samples=5664 | coefficients=280 | groups=140
  residual radius=1.328e-10 | relative target=2.411e-11
  backend=dense-svd Douglas-Rachford | rank=223/280 | DR step=1.034e-03
  support selection=disabled here; solver returns group ranking only
  [iter      1] fixed-point=9.981e-01 | rel-res=2.411e-11
  [iter    500] fixed-point=6.097e-04 | rel-res=2.411e-11
  [iter   1000] fixed-point=1.213e-04 | rel-res=2.411e-11
  [iter   1500] fixed-point=1.042e-06 | rel-res=2.411e-11
  [iter   2000] fixed-point=5.888e-08 | rel-res=2.411e-11
  [iter   2500] fixed-point=1.327e-08 | rel-res=2.411e-11
Grouped basis-pursuit denoising complete.
  stop reason=Douglas-Rachford fixed-point tolerance reached
  iterations=2650 | fixed-point=9.196e-09 | feasible=True
  relative residual=2.411e-11 | objective=2.286140e+01
Starting ranked-prefix support certification...
  candidate groups=140 | prefix cap=140 | relative uncertainty floor=2.411e

In [11]:
# ============================================================
# Cell 10 — TIDES STEP 3 (BLIND):
# STAGE-WISE VECTOR-FIELD RECONSTRUCTION
#
# Input:
#     Step-2 blind changed-edge supports
#
# Reconstruct jointly:
#
#     B^(1)
#     Delta B^(1), ..., Delta B^(K)
#
# then form
#
#     B^(r) = B^(1) + sum_{k<r} Delta B^(k)
#
# No oracle topology, weights, or interaction law are used.
# ============================================================

import importlib
import solvers_linear_regression
import step3_vector_field

# Development reload after replacing the modules on disk.
importlib.reload(solvers_linear_regression)
importlib.reload(step3_vector_field)

from step3_vector_field import (
    reconstruct_vector_field_from_observations,
)


# ------------------------------------------------------------
# 1. Formal blind Step-3 reconstruction
#
# STEP2_RESULT is passed directly.
# Step 3 uses ONLY its selected supports.
#
# Step-2 projected coefficients are NOT reused.
# ------------------------------------------------------------

STEP3_RESULT = reconstruct_vector_field_from_observations(
    Y=V_MID,
    D=D,
    edge_features=EDGE_PSI,
    stage_of_sample=OBS_STAGE,

    change_constraints_or_supports=STEP2_RESULT,

    # N=8 correctness benchmark:
    # use explicit dense SVD least squares.
    solver_method="dense_lstsq",

    solver_kwargs={
        "rcond": None,
        "compute_raw_svd_diagnostics": True,
        "verbose": True,
    },

    design_mode="dense",

    # Keep design for notebook-level structural diagnostics.
    return_design=True,
)


# ------------------------------------------------------------
# 2. Basic dimensions
# ------------------------------------------------------------

design = STEP3_RESULT.design
solver = STEP3_RESULT.solver_result

selected_change_groups = sum(
    len(c.support)
    for c in STEP2_RESULT.constraints
)

expected_parameter_count = (
    M * L
    + selected_change_groups * L
)


# ------------------------------------------------------------
# 3. Formal blind diagnostics
# ------------------------------------------------------------

print()
print("=" * 90)
print("Cell 10 — TIDES STEP 3 (BLIND): RESULT")
print("=" * 90)

print(
    "preprocessed observations :",
    STEP3_RESULT.n_preprocessed_observations,
)

print(
    "scalar observations       :",
    STEP3_RESULT.n_observations,
)

print(
    "selected change groups    :",
    selected_change_groups,
)

print(
    "anchor parameters         :",
    M * L,
)

print(
    "change parameters         :",
    selected_change_groups * L,
)

print(
    "total parameters          :",
    STEP3_RESULT.parameter_count,
)

print(
    "design shape              :",
    design.matrix.shape,
)

print(
    "solver method             :",
    solver.method,
)

print(
    "rank                      :",
    f"{STEP3_RESULT.rank}/{STEP3_RESULT.parameter_count}",
)

print(
    "identifiable              :",
    STEP3_RESULT.identifiable,
)

print(
    "condition number (scaled) :",
    f"{STEP3_RESULT.condition_number_scaled:.6e}",
)

print(
    "condition number (raw)    :",
    f"{STEP3_RESULT.condition_number_raw:.6e}",
)

print(
    "relative residual         :",
    f"{STEP3_RESULT.relative_residual:.6e}",
)

print(
    "normal-equation residual  :",
    f"{solver.normal_equation_relative_residual:.6e}",
)

print(
    "profile floor             :",
    f"{PROFILE_FLOOR:.6e}",
)

print(
    "below data resolution     :",
    STEP3_RESULT.relative_residual <= PROFILE_FLOOR,
)

print(
    "B_anchor shape            :",
    STEP3_RESULT.B_anchor.shape,
)

print(
    "number of Delta B blocks  :",
    len(STEP3_RESULT.delta_B),
)

print(
    "B_stages shape            :",
    STEP3_RESULT.B_stages.shape,
)

print()
print("oracle information used   : NO")


# ------------------------------------------------------------
# 4. Internal consistency assertions
#
# These assertions use inferred structure only.
# No benchmark truth enters here.
# ------------------------------------------------------------

assert STEP3_RESULT.parameter_count == expected_parameter_count

assert design.matrix.shape == (
    len(V_MID) * N,
    expected_parameter_count,
)

assert STEP3_RESULT.B_anchor.shape == (M, L)

assert len(STEP3_RESULT.delta_B) == len(
    STEP2_RESULT.constraints
)

assert all(
    dB.shape == (M, L)
    for dB in STEP3_RESULT.delta_B
)

assert STEP3_RESULT.B_stages.shape == (
    len(STEP2_RESULT.constraints) + 1,
    M,
    L,
)

assert np.all(np.isfinite(STEP3_RESULT.B_anchor))
assert np.all(np.isfinite(STEP3_RESULT.B_stages))
assert np.all(np.isfinite(STEP3_RESULT.residual_vector_field))

assert solver.converged

if STEP3_RESULT.rank is not None:
    assert STEP3_RESULT.rank == STEP3_RESULT.parameter_count

assert STEP3_RESULT.relative_residual <= PROFILE_FLOOR


print()
print("TIDES STEP 3 RECONSTRUCTION: PASS")

Starting linear least-squares solve...
  samples=5664 | features=96 | outputs=1 | method=dense_lstsq
Linear least-squares solve complete.
  relative residual=8.467e-14 | normal-eq residual=1.273e-02
  rank=96/96 | cond(scaled)=4.271e+04 | cond(raw)=2.696e+05

Cell 10 — TIDES STEP 3 (BLIND): RESULT
preprocessed observations : 708
scalar observations       : 5664
selected change groups    : 20
anchor parameters         : 56
change parameters         : 40
total parameters          : 96
design shape              : (5664, 96)
solver method             : dense_lstsq
rank                      : 96/96
identifiable              : True
condition number (scaled) : 4.271319e+04
condition number (raw)    : 2.696226e+05
relative residual         : 8.467042e-14
normal-equation residual  : 1.273398e-02
profile floor             : 2.411322e-11
below data resolution     : True
B_anchor shape            : (28, 2)
number of Delta B blocks  : 5
B_stages shape            : (6, 28, 2)

oracle information use

In [12]:
# ============================================================
# Cell 10A — TIDES STEP 3 VALIDATION:
# VECTOR-FIELD COEFFICIENT RECOVERY
#
# ORACLE / BENCHMARK VALIDATION ONLY.
#
# Compare the BLIND Step-3 reconstruction against the known
# benchmark vector fields.
#
# Truth is used only below this line; it was not used by
# Steps 1–3 inference.
# ============================================================


# ------------------------------------------------------------
# 1. Construct true stage-wise B matrices
#
# For an active edge m:
#
#     B_m = w_m * [1, 0.5]
#
# Inactive edges have zero rows.
# ------------------------------------------------------------

B_TRUE_STAGES = np.zeros(
    (N_STAGES, M, L),
    dtype=float,
)

for r, snapshot in enumerate(SNAPSHOTS):
    for edge in snapshot:
        m = EDGE_INDEX_1B[edge]

        B_TRUE_STAGES[r, m, :] = (
            EDGE_WEIGHT[edge] * PAIR_LAW_TRUE
        )


DELTA_B_TRUE = np.diff(
    B_TRUE_STAGES,
    axis=0,
)


# ------------------------------------------------------------
# 2. Reconstructed quantities
# ------------------------------------------------------------

B_HAT_STAGES = STEP3_RESULT.B_stages

DELTA_B_HAT = np.stack(
    STEP3_RESULT.delta_B,
    axis=0,
)


# ------------------------------------------------------------
# 3. Global coefficient errors
# ------------------------------------------------------------

anchor_relative_error = (
    np.linalg.norm(
        STEP3_RESULT.B_anchor - B_TRUE_STAGES[0]
    )
    /
    np.linalg.norm(B_TRUE_STAGES[0])
)

stage_relative_error = (
    np.linalg.norm(
        B_HAT_STAGES - B_TRUE_STAGES
    )
    /
    np.linalg.norm(B_TRUE_STAGES)
)

delta_relative_error = (
    np.linalg.norm(
        DELTA_B_HAT - DELTA_B_TRUE
    )
    /
    np.linalg.norm(DELTA_B_TRUE)
)

max_abs_stage_error = np.max(
    np.abs(
        B_HAT_STAGES - B_TRUE_STAGES
    )
)

max_abs_delta_error = np.max(
    np.abs(
        DELTA_B_HAT - DELTA_B_TRUE
    )
)


# ------------------------------------------------------------
# 4. Per-stage / per-transition errors
# ------------------------------------------------------------

per_stage_errors = []

for r in range(N_STAGES):

    truth_norm = np.linalg.norm(
        B_TRUE_STAGES[r]
    )

    err = (
        np.linalg.norm(
            B_HAT_STAGES[r]
            - B_TRUE_STAGES[r]
        )
        /
        max(truth_norm, np.finfo(float).tiny)
    )

    per_stage_errors.append(float(err))


per_transition_errors = []

for k in range(N_STAGES - 1):

    truth_norm = np.linalg.norm(
        DELTA_B_TRUE[k]
    )

    err = (
        np.linalg.norm(
            DELTA_B_HAT[k]
            - DELTA_B_TRUE[k]
        )
        /
        max(truth_norm, np.finfo(float).tiny)
    )

    per_transition_errors.append(float(err))


# ------------------------------------------------------------
# 5. Leakage onto coefficients that are truly zero
# ------------------------------------------------------------

true_zero_mask = np.isclose(
    B_TRUE_STAGES,
    0.0,
    atol=0.0,
)

max_inactive_coefficient = np.max(
    np.abs(
        B_HAT_STAGES[true_zero_mask]
    )
)


# ------------------------------------------------------------
# 6. Report
# ------------------------------------------------------------

print("=" * 90)
print("Cell 10A — TIDES STEP 3 VALIDATION")
print("=" * 90)

print(
    "anchor relative error      :",
    f"{anchor_relative_error:.6e}",
)

print(
    "all-stage relative error   :",
    f"{stage_relative_error:.6e}",
)

print(
    "all-delta relative error   :",
    f"{delta_relative_error:.6e}",
)

print(
    "max |B_hat - B_true|       :",
    f"{max_abs_stage_error:.6e}",
)

print(
    "max |dB_hat - dB_true|     :",
    f"{max_abs_delta_error:.6e}",
)

print(
    "max inactive coefficient   :",
    f"{max_inactive_coefficient:.6e}",
)

print()

for r, err in enumerate(per_stage_errors):
    print(
        f"stage {r + 1} relative error     : "
        f"{err:.6e}"
    )

print()

for k, err in enumerate(per_transition_errors):
    print(
        f"transition {k + 1} delta error  : "
        f"{err:.6e}"
    )


# ------------------------------------------------------------
# 7. Benchmark validation assertions
#
# Tolerance is intentionally much looser than observed numerical
# accuracy. This is a correctness regression threshold, not an
# inference hyperparameter.
# ------------------------------------------------------------

VALIDATION_TOL = 1e-7

assert anchor_relative_error < VALIDATION_TOL
assert stage_relative_error < VALIDATION_TOL
assert delta_relative_error < VALIDATION_TOL

print()
print("ORACLE INFORMATION USED: YES — VALIDATION ONLY")
print("TIDES STEP 3 COEFFICIENT RECOVERY: PASS")

Cell 10A — TIDES STEP 3 VALIDATION
anchor relative error      : 7.206851e-10
all-stage relative error   : 7.672341e-10
all-delta relative error   : 7.742345e-10
max |B_hat - B_true|       : 1.943273e-09
max |dB_hat - dB_true|     : 3.350517e-09
max inactive coefficient   : 1.407244e-09

stage 1 relative error     : 7.206851e-10
stage 2 relative error     : 7.430369e-10
stage 3 relative error     : 7.422235e-10
stage 4 relative error     : 8.089582e-10
stage 5 relative error     : 8.110437e-10
stage 6 relative error     : 7.694178e-10

transition 1 delta error  : 2.300004e-10
transition 2 delta error  : 1.965865e-10
transition 3 delta error  : 1.552881e-09
transition 4 delta error  : 7.251374e-10
transition 5 delta error  : 4.113836e-10

ORACLE INFORMATION USED: YES — VALIDATION ONLY
TIDES STEP 3 COEFFICIENT RECOVERY: PASS


In [13]:
# ============================================================
# Cell 11 — TIDES STEP 4 (BLIND):
# SHARED-LAW SOURCE DECOMPOSITION
#
# Hypothesis:
#
#     B^(r) = W^(r) theta^T
#
# Input:
#     Blind Step-3 reconstructed B_stages
#
# Output:
#     theta       : shared interaction-law coefficients
#     W_stages    : stage-wise microscopic edge amplitudes
#
# No true interaction law, true edge weights, or true snapshots
# are used in this cell.
# ============================================================

import importlib
import step4_source_decomposition

importlib.reload(step4_source_decomposition)

from step4_source_decomposition import (
    decompose_vector_field_sources,
)


# ------------------------------------------------------------
# 1. Formal blind Step-4 decomposition
#
# Gauge convention:
#
#     theta[0] = 1
#
# This only fixes the intrinsic scale ambiguity
#
#     W -> c W
#     theta -> theta / c
#
# and does NOT use PAIR_LAW_TRUE.
# ------------------------------------------------------------

STEP4_RESULT = decompose_vector_field_sources(
    STEP3_RESULT,

    hypothesis="shared_interaction_law",

    normalization="reference_component",
    reference_component=0,
)


# ------------------------------------------------------------
# 2. Basic decomposition output
# ------------------------------------------------------------

print("=" * 90)
print("Cell 11 — TIDES STEP 4 (BLIND): RESULT")
print("=" * 90)

print(
    "B_stages shape            :",
    STEP3_RESULT.B_stages.shape,
)

print(
    "W_stages shape            :",
    STEP4_RESULT.W_stages.shape,
)

print(
    "theta shape               :",
    STEP4_RESULT.theta.shape,
)

print()

print(
    "inferred theta            :",
    np.array2string(
        STEP4_RESULT.theta,
        precision=12,
        suppress_small=False,
    ),
)

print(
    "unit-gauge theta          :",
    np.array2string(
        STEP4_RESULT.theta_unit,
        precision=12,
        suppress_small=False,
    ),
)

print(
    "gauge normalization       :",
    STEP4_RESULT.normalization,
)

print(
    "gauge component           :",
    STEP4_RESULT.gauge_component,
)


# ------------------------------------------------------------
# 3. Rank-one / shared-law diagnostics
# ------------------------------------------------------------

print()
print("-" * 90)
print("SHARED-LAW DIAGNOSTICS")
print("-" * 90)

print(
    "singular values           :",
    np.array2string(
        STEP4_RESULT.singular_values,
        precision=12,
        suppress_small=False,
    ),
)

print(
    "numerical SVD rank        :",
    STEP4_RESULT.numerical_rank,
)

print(
    "rank tolerance            :",
    f"{STEP4_RESULT.rank_tolerance:.6e}",
)

print(
    "rank-1 energy fraction    :",
    f"{STEP4_RESULT.rank1_energy_fraction:.16f}",
)

print(
    "s2 / s1                   :",
    f"{STEP4_RESULT.second_to_first_singular_ratio:.6e}",
)

print(
    "spectral gap s1 / s2      :",
    f"{STEP4_RESULT.spectral_gap:.6e}",
)

print(
    "rank-1 relative residual  :",
    f"{STEP4_RESULT.relative_residual:.6e}",
)


# ------------------------------------------------------------
# 4. Stage-wise rank-one residuals
# ------------------------------------------------------------

print()
print("-" * 90)
print("STAGE-WISE DECOMPOSITION RESIDUALS")
print("-" * 90)

for r, err in enumerate(
    STEP4_RESULT.relative_residual_by_stage
):
    print(
        f"stage {r + 1} relative residual : "
        f"{err:.6e}"
    )


# ------------------------------------------------------------
# 5. Internal consistency checks
#
# These checks use only inferred quantities.
# ------------------------------------------------------------

B_FROM_FACTORS = (
    STEP4_RESULT.W_stages[..., None]
    * STEP4_RESULT.theta[None, None, :]
)

factorization_consistency = (
    np.linalg.norm(
        B_FROM_FACTORS
        - STEP4_RESULT.B_reconstructed
    )
    /
    max(
        np.linalg.norm(
            STEP4_RESULT.B_reconstructed
        ),
        np.finfo(float).tiny,
    )
)

print()
print("-" * 90)
print("INTERNAL CONSISTENCY")
print("-" * 90)

print(
    "factor reconstruction err :",
    f"{factorization_consistency:.6e}",
)

print(
    "theta[0] after gauge fix  :",
    f"{STEP4_RESULT.theta[0]:.12f}",
)

print()
print("oracle information used   : NO")


# ------------------------------------------------------------
# 6. Assertions
#
# No oracle accuracy threshold is used here.
# ------------------------------------------------------------

R, M_, L_ = STEP3_RESULT.B_stages.shape

assert STEP4_RESULT.W_stages.shape == (R, M_)
assert STEP4_RESULT.theta.shape == (L_,)
assert STEP4_RESULT.B_reconstructed.shape == (R, M_, L_)
assert STEP4_RESULT.residual_B.shape == (R, M_, L_)

assert np.all(np.isfinite(STEP4_RESULT.theta))
assert np.all(np.isfinite(STEP4_RESULT.W_stages))
assert np.all(np.isfinite(STEP4_RESULT.B_reconstructed))

assert np.isclose(
    STEP4_RESULT.theta[0],
    1.0,
    rtol=1e-12,
    atol=1e-12,
)

assert factorization_consistency < 1e-12

print()
print("TIDES STEP 4 SOURCE DECOMPOSITION: PASS")

Cell 11 — TIDES STEP 4 (BLIND): RESULT
B_stages shape            : (6, 28, 2)
W_stages shape            : (6, 28)
theta shape               : (2,)

inferred theta            : [1.             0.500000000005]
unit-gauge theta          : [0.894427190998 0.447213595503]
gauge normalization       : reference_component
gauge component           : 0

------------------------------------------------------------------------------------------
SHARED-LAW DIAGNOSTICS
------------------------------------------------------------------------------------------
singular values           : [8.356954962733e+00 5.347253870851e-09]
numerical SVD rank        : 2
rank tolerance            : 3.117436e-13
rank-1 energy fraction    : 1.0000000000000000
s2 / s1                   : 6.398567e-10
spectral gap s1 / s2      : 1.562850e+09
rank-1 relative residual  : 6.398567e-10

------------------------------------------------------------------------------------------
STAGE-WISE DECOMPOSITION RESIDUALS
------------

In [14]:
# ============================================================
# Cell 11A — TIDES STEP 4 VALIDATION:
# MICROSCOPIC SOURCE AND INTERACTION-LAW RECOVERY
#
# ORACLE / BENCHMARK VALIDATION ONLY.
#
# Compare the BLIND Step-4 decomposition against:
#
#     theta_true = PAIR_LAW_TRUE
#     W_true^(r) = true stage-wise edge weights
#
# Truth is used only in this validation cell.
# It was not used by Steps 1–4 inference.
# ============================================================


# ------------------------------------------------------------
# 1. Construct true microscopic source amplitudes
#
# For every stage r and candidate edge m:
#
#     W_true[r,m] = EDGE_WEIGHT[edge]   if edge is active
#                 = 0                   otherwise
# ------------------------------------------------------------

W_TRUE_STAGES = np.zeros(
    (N_STAGES, M),
    dtype=float,
)

for r, snapshot in enumerate(SNAPSHOTS):
    for edge in snapshot:
        m = EDGE_INDEX_1B[edge]
        W_TRUE_STAGES[r, m] = EDGE_WEIGHT[edge]


THETA_TRUE = np.asarray(
    PAIR_LAW_TRUE,
    dtype=float,
)


# ------------------------------------------------------------
# 2. Blind Step-4 estimates
# ------------------------------------------------------------

THETA_HAT = np.asarray(
    STEP4_RESULT.theta,
    dtype=float,
)

W_HAT_STAGES = np.asarray(
    STEP4_RESULT.W_stages,
    dtype=float,
)


# ------------------------------------------------------------
# 3. Interaction-law recovery error
# ------------------------------------------------------------

theta_absolute_error = np.linalg.norm(
    THETA_HAT - THETA_TRUE
)

theta_relative_error = (
    theta_absolute_error
    /
    np.linalg.norm(THETA_TRUE)
)

theta_max_abs_error = np.max(
    np.abs(
        THETA_HAT - THETA_TRUE
    )
)


# ------------------------------------------------------------
# 4. Microscopic source-amplitude recovery error
# ------------------------------------------------------------

W_relative_error = (
    np.linalg.norm(
        W_HAT_STAGES - W_TRUE_STAGES
    )
    /
    np.linalg.norm(W_TRUE_STAGES)
)

W_max_abs_error = np.max(
    np.abs(
        W_HAT_STAGES - W_TRUE_STAGES
    )
)


# ------------------------------------------------------------
# 5. Stage-wise source recovery
# ------------------------------------------------------------

per_stage_W_errors = []

for r in range(N_STAGES):

    truth_norm = np.linalg.norm(
        W_TRUE_STAGES[r]
    )

    err = (
        np.linalg.norm(
            W_HAT_STAGES[r]
            - W_TRUE_STAGES[r]
        )
        /
        max(
            truth_norm,
            np.finfo(float).tiny,
        )
    )

    per_stage_W_errors.append(
        float(err)
    )


# ------------------------------------------------------------
# 6. Leakage onto truly inactive microscopic edges
# ------------------------------------------------------------

inactive_mask = np.isclose(
    W_TRUE_STAGES,
    0.0,
    atol=0.0,
)

active_mask = ~inactive_mask

max_inactive_W = np.max(
    np.abs(
        W_HAT_STAGES[inactive_mask]
    )
)

active_W_relative_error = (
    np.linalg.norm(
        W_HAT_STAGES[active_mask]
        - W_TRUE_STAGES[active_mask]
    )
    /
    np.linalg.norm(
        W_TRUE_STAGES[active_mask]
    )
)


# ------------------------------------------------------------
# 7. Full microscopic reconstruction consistency
#
# Reconstruct B from the recovered microscopic sources:
#
#     B_micro^(r) = W_hat^(r) theta_hat^T
#
# and compare directly with the true benchmark B.
# ------------------------------------------------------------

B_MICRO_HAT = (
    W_HAT_STAGES[..., None]
    * THETA_HAT[None, None, :]
)

B_TRUE_FROM_SOURCES = (
    W_TRUE_STAGES[..., None]
    * THETA_TRUE[None, None, :]
)

microscopic_B_relative_error = (
    np.linalg.norm(
        B_MICRO_HAT
        - B_TRUE_FROM_SOURCES
    )
    /
    np.linalg.norm(
        B_TRUE_FROM_SOURCES
    )
)


# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print("=" * 90)
print("Cell 11A — TIDES STEP 4 VALIDATION")
print("=" * 90)

print(
    "theta true               :",
    np.array2string(
        THETA_TRUE,
        precision=12,
        suppress_small=False,
    ),
)

print(
    "theta inferred           :",
    np.array2string(
        THETA_HAT,
        precision=12,
        suppress_small=False,
    ),
)

print(
    "theta absolute error     :",
    f"{theta_absolute_error:.6e}",
)

print(
    "theta relative error     :",
    f"{theta_relative_error:.6e}",
)

print(
    "theta max abs error      :",
    f"{theta_max_abs_error:.6e}",
)

print()

print(
    "all-W relative error     :",
    f"{W_relative_error:.6e}",
)

print(
    "active-W relative error  :",
    f"{active_W_relative_error:.6e}",
)

print(
    "max |W_hat - W_true|     :",
    f"{W_max_abs_error:.6e}",
)

print(
    "max inactive W           :",
    f"{max_inactive_W:.6e}",
)

print(
    "microscopic B rel. error :",
    f"{microscopic_B_relative_error:.6e}",
)

print()

for r, err in enumerate(
    per_stage_W_errors
):
    print(
        f"stage {r + 1} W relative error   : "
        f"{err:.6e}"
    )


# ------------------------------------------------------------
# 9. Benchmark validation assertions
#
# These tolerances are correctness-regression thresholds only.
# They are not inference hyperparameters.
# ------------------------------------------------------------

VALIDATION_TOL = 1e-7

assert theta_relative_error < VALIDATION_TOL
assert W_relative_error < VALIDATION_TOL
assert microscopic_B_relative_error < VALIDATION_TOL

print()
print("ORACLE INFORMATION USED: YES — VALIDATION ONLY")
print("TIDES STEP 4 SOURCE RECOVERY: PASS")

Cell 11A — TIDES STEP 4 VALIDATION
theta true               : [1.  0.5]
theta inferred           : [1.             0.500000000005]
theta absolute error     : 4.718781e-12
theta relative error     : 4.220606e-12
theta max abs error      : 4.718781e-12

all-W relative error     : 4.233719e-10
active-W relative error  : 2.841260e-10
max |W_hat - W_true|     : 8.770790e-10
max inactive W           : 8.770790e-10
microscopic B rel. error : 4.233574e-10

stage 1 W relative error   : 4.317620e-10
stage 2 W relative error   : 4.474641e-10
stage 3 W relative error   : 4.504272e-10
stage 4 W relative error   : 4.190166e-10
stage 5 W relative error   : 4.117184e-10
stage 6 W relative error   : 3.743616e-10

ORACLE INFORMATION USED: YES — VALIDATION ONLY
TIDES STEP 4 SOURCE RECOVERY: PASS
